# Full Wikipedia Article Extraction

Extracts the full plain-text Wikipedia article for every politician in `politicians_network_nodes2.csv`.

Uses MediaWiki extracts API. Caches each article to `wiki_full_text_cache/<wikidata>.json`. Errored cache entries are automatically retried on re-run.

Output: `politicians_full_text.csv` with the original metadata columns plus one text column `body_text` containing the full cleaned article (intro + all kept sections, with `== Section ==` markers preserved). Boilerplate footer sections (References, See also, External links, Bibliography, Sources, Further reading, Notes, Footnotes, Citations, Works, Publications) are dropped together with their subsections.

Final columns:
`wikidata, name, position, start, end, nationality, birth_date, death_date, party, party_simple, gender, education, state, degree, wikipedia_url, body_text`

In [1]:
import os, re, json, time, glob
from urllib.parse import unquote
import requests
import pandas as pd
from tqdm import tqdm

API_URL = 'https://en.wikipedia.org/w/api.php'
CACHE_DIR = 'wiki_full_text_cache'
INPUT_CSV = 'politicians_with_communities.csv'
OUTPUT_CSV = 'politicians_full_text.csv'
REQUEST_DELAY_S = 0.15
REQUEST_TIMEOUT_S = 30

os.makedirs(CACHE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({'User-Agent': 'PoliticiansNetwork/1.0 (s245290@dtu.dk) full-article extraction'})

In [2]:
df_nodes = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df_nodes)} politicians from {INPUT_CSV}')
print('Columns:', list(df_nodes.columns))
df_nodes.head(3)

Loaded 6138 politicians from politicians_with_communities.csv
Columns: ['wikidata', 'name', 'position', 'nationality', 'birth_date', 'death_date', 'party', 'gender', 'education', 'state', 'wikipedia_url', 'career_start', 'career_end', 'connections', 'is_seed', 'prestigious_school', 'degree', 'community']


,wikidata,name,position,nationality,birth_date,death_date,party,gender,education,state,wikipedia_url,career_start,career_end,connections,is_seed,prestigious_school,degree,community
0,Q27996060,Stephen Miller,Senior Advisor to the President of the United ...,United States,1985-08-23,NaN,Republican,male,Santa Monica High School,NaN,https://en.wikipedia.org/wiki/Stephen_Miller,2017-01-20,2021-01-20,"['Q160582', 'Q1701732', 'Q358443', 'Q76', 'Q16...",False,False,41.0,2.0
1,Q259514,Valerie Jarrett,Senior Advisor to the President of the United ...,United States,1956-11-14,NaN,Democrat,female,Stanford University,NaN,https://en.wikipedia.org/wiki/Valerie_Jarrett,2009-01-20,2017-01-20,"['Q76', 'Q295584', 'Q539814', 'Q13133', 'Q8274...",False,True,15.0,22.0
2,Q561284,Cedric Richmond,Senior Advisor to the President of the United ...,United States,1973-09-13,NaN,Democrat,male,Tulane University Law School,NaN,https://en.wikipedia.org/wiki/Cedric_Richmond,1999-01-01,2022-05-18,"['Q15304910', 'Q5703668', 'Q336324', 'Q76', 'Q...",True,False,11.0,17.0


## API helper

Uses `prop=extracts` with `explaintext=1` (clean plain text, no HTML/refs/infoboxes) and `exsectionformat=wiki` (preserves `== Section ==` markers). One title per call so we get full text (batching forces intro-only).

In [3]:
def title_from_url(url):
    if not isinstance(url, str) or '/wiki/' not in url:
        return ''
    return unquote(url.rstrip('/').split('/wiki/')[-1])

def fetch_full_article(title):
    if not title:
        return ''
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'extracts',
        'explaintext': 1,
        'exsectionformat': 'wiki',
        'redirects': 1,
    }
    r = session.get(API_URL, params=params, timeout=REQUEST_TIMEOUT_S)
    r.raise_for_status()
    pages = r.json().get('query', {}).get('pages', {})
    if not pages:
        return ''
    page = next(iter(pages.values()))
    return page.get('extract', '') or ''

## Boilerplate filter

`clean_full_text` removes top-level sections (and their subsections) matching: References, Notes, Footnotes, Citations, See also, External links, Bibliography, Sources, Further reading, Works, Selected works, Publications, Selected publications. Returns the full cleaned article as one string with `== Section ==` markers preserved.

In [4]:
SECTION_HEADER_RE = re.compile(r'^(=+)\s*(.+?)\s*\1\s*$', re.MULTILINE)

BOILERPLATE_PATTERNS = [
    'references', 'notes', 'footnotes', 'citations',
    'see also', 'external links',
    'bibliography', 'sources', 'further reading',
    'works', 'selected works', 'publications', 'selected publications',
]

def is_boilerplate(section_title):
    t = section_title.strip().lower().rstrip(':')
    if t in BOILERPLATE_PATTERNS:
        return True
    for p in BOILERPLATE_PATTERNS:
        if t == p or t.startswith(p + ' ') or t.endswith(' ' + p):
            return True
    return False

def clean_full_text(full_text):
    """Return the full article as one string with boilerplate sections removed.
    Keeps the intro plus all non-boilerplate sections, with `== Section ==` headers preserved."""
    if not full_text:
        return ''
    matches = list(SECTION_HEADER_RE.finditer(full_text))
    if not matches:
        return full_text.strip()
    intro = full_text[:matches[0].start()].strip()
    sections = []
    for i, m in enumerate(matches):
        level = len(m.group(1))
        title = m.group(2).strip()
        s = m.end()
        e = matches[i+1].start() if i+1 < len(matches) else len(full_text)
        sections.append((level, title, full_text[s:e].strip()))
    cleaned_parts = []
    if intro:
        cleaned_parts.append(intro)
    skipping = False
    for level, title, body in sections:
        if level == 2:
            skipping = is_boilerplate(title)
        if skipping:
            continue
        marker = '=' * level
        cleaned_parts.append(f'{marker} {title} {marker}')
        if body:
            cleaned_parts.append(body)
    return '\n\n'.join(cleaned_parts).strip()

In [5]:
_demo = (
    'Barack Obama is a politician.\n\n'
    '== Early life and education ==\n\nBorn in Honolulu.\n\n'
    '=== Childhood ===\n\nHawaii.\n\n'
    '== Political career ==\n\nServed as senator.\n\n'
    '== References ==\n\n[1] cite\n\n'
    '=== Sub-ref ===\n\nshould drop\n\n'
    '== External links ==\n\nlinks\n'
)
print(clean_full_text(_demo))

Barack Obama is a politician.

== Early life and education ==

Born in Honolulu.

=== Childhood ===

Hawaii.

== Political career ==

Served as senator.


## Cache helpers — successful entries reused, errored entries retried

In [6]:
def cache_path(wikidata_id):
    return os.path.join(CACHE_DIR, f'{wikidata_id}.json')

def load_cached(wikidata_id):
    p = cache_path(wikidata_id)
    if os.path.exists(p):
        try:
            with open(p, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception:
            return None
    return None

def save_cached(wikidata_id, payload):
    with open(cache_path(wikidata_id), 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False)

def is_usable_cache(payload):
    return payload is not None and not payload.get('error')

def get_or_fetch(wikidata_id, title):
    cached = load_cached(wikidata_id)
    if is_usable_cache(cached):
        return cached, True
    try:
        text = fetch_full_article(title)
        payload = {'wikidata': wikidata_id, 'title': title, 'raw_text': text, 'error': None}
    except Exception as e:
        payload = {'wikidata': wikidata_id, 'title': title, 'raw_text': '', 'error': str(e)}
    save_cached(wikidata_id, payload)
    return payload, False

In [7]:
_existing = glob.glob(os.path.join(CACHE_DIR, '*.json'))
_err = 0
_ok = 0
for f in _existing:
    try:
        d = json.load(open(f, encoding='utf-8'))
        if d.get('error'):
            _err += 1
        else:
            _ok += 1
    except Exception:
        pass
print(f'Cache files: {len(_existing)} ({_ok} OK, {_err} errored — will be retried)')

Cache files: 6138 (6138 OK, 0 errored — will be retried)


## Main extraction loop

In [8]:
rows = []
errors = []
fetched_from_api = 0
loaded_from_cache = 0

subset = df_nodes[df_nodes['wikipedia_url'].notna() & (df_nodes['wikipedia_url'] != '')].copy()
print(f'Processing {len(subset)} politicians.')

for _, row in tqdm(subset.iterrows(), total=len(subset), desc='Fetching full articles'):
    wikidata_id = row['wikidata']
    title = title_from_url(row['wikipedia_url'])
    payload, was_cached = get_or_fetch(wikidata_id, title)
    if was_cached:
        loaded_from_cache += 1
    else:
        fetched_from_api += 1
        time.sleep(REQUEST_DELAY_S)
    if payload.get('error'):
        errors.append((wikidata_id, title, payload['error']))
    body_text = clean_full_text(payload.get('raw_text', ''))
    rows.append({'wikidata': wikidata_id, 'body_text': body_text})

print(f'\nDone. From cache: {loaded_from_cache} | from API: {fetched_from_api} | errors: {len(errors)}')
if errors:
    print('First 10 errors:')
    for e in errors[:10]:
        print(' ', e[0], e[1], '->', e[2][:120])

Processing 6138 politicians.


Fetching full articles: 100%|██████████| 6138/6138 [00:01<00:00, 3353.56it/s]


Done. From cache: 6136 | from API: 2 | errors: 0


## Build wide CSV — metadata + body_text

Renames `career_start` → `start` and `career_end` → `end`.

In [9]:
df_text = pd.DataFrame(rows)
df_meta = df_nodes.rename(columns={'career_start': 'start', 'career_end': 'end'}).copy()
df_out = df_meta.merge(df_text, on='wikidata', how='left')


df_out.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df_out)} rows to {OUTPUT_CSV}')
print('Columns:', list(df_out.columns))
df_out.head(3)

Saved 6138 rows to politicians_full_text.csv
Columns: ['wikidata', 'name', 'position', 'nationality', 'birth_date', 'death_date', 'party', 'gender', 'education', 'state', 'wikipedia_url', 'start', 'end', 'connections', 'is_seed', 'prestigious_school', 'degree', 'community', 'body_text']


,wikidata,name,position,nationality,birth_date,death_date,party,gender,education,state,wikipedia_url,start,end,connections,is_seed,prestigious_school,degree,community,body_text
0,Q27996060,Stephen Miller,Senior Advisor to the President of the United ...,United States,1985-08-23,NaN,Republican,male,Santa Monica High School,NaN,https://en.wikipedia.org/wiki/Stephen_Miller,2017-01-20,2021-01-20,"['Q160582', 'Q1701732', 'Q358443', 'Q76', 'Q16...",False,False,41.0,2.0,"Stephen N. Miller (born August 23, 1985) is an..."
1,Q259514,Valerie Jarrett,Senior Advisor to the President of the United ...,United States,1956-11-14,NaN,Democrat,female,Stanford University,NaN,https://en.wikipedia.org/wiki/Valerie_Jarrett,2009-01-20,2017-01-20,"['Q76', 'Q295584', 'Q539814', 'Q13133', 'Q8274...",False,True,15.0,22.0,Valerie June Jarrett (née Bowman; born Novembe...
2,Q561284,Cedric Richmond,Senior Advisor to the President of the United ...,United States,1973-09-13,NaN,Democrat,male,Tulane University Law School,NaN,https://en.wikipedia.org/wiki/Cedric_Richmond,1999-01-01,2022-05-18,"['Q15304910', 'Q5703668', 'Q336324', 'Q76', 'Q...",True,False,11.0,17.0,"Cedric Levan Richmond (born September 13, 1973..."


## Sanity checks

In [10]:
body_len = df_out['body_text'].fillna('').astype(str).str.len()

print('=== Coverage ===')
print(f'Rows total              : {len(df_out)}')
print(f'Empty body_text         : {(body_len == 0).sum()}')
print()
print('=== body_text length distribution (chars) ===')
print(body_len.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(0))
print()
print('=== Top 5 longest ===')
tmp = df_out.assign(_len=body_len)
print(tmp.nlargest(5, '_len')[['name', '_len']].to_string(index=False))
print()
print('=== 5 shortest non-empty ===')
ne = tmp[tmp['_len'] > 0]
print(ne.nsmallest(5, '_len')[['name', '_len']].to_string(index=False))
print()
print('=== Spot check: did Obama, Biden, Trump come back? ===')
for q in ['Q76', 'Q6279', 'Q22686']:
    sub = df_out[df_out['wikidata'] == q]
    if len(sub):
        r = sub.iloc[0]
        print(f"  {r['name']:20s} body_len={len(str(r['body_text'])):>7d}")

=== Coverage ===
Rows total              : 6138
Empty body_text         : 0

=== body_text length distribution (chars) ===
count      6138.0
mean       9206.0
std       12282.0
min          83.0
5%          739.0
25%        2391.0
50%        5134.0
75%       10823.0
95%       32276.0
max      113268.0
Name: body_text, dtype: float64

=== Top 5 longest ===
                 name   _len
        Rudy Giuliani 113268
        Ralph Northam 106902
          Robert Byrd 105630
Robert F. Kennedy Jr.  99600
         Nancy Pelosi  98048

=== 5 shortest non-empty ===
            name  _len
Thayer Verschoor    83
    Daniel Beren   124
 Terry John Care   129
Barbara Friedman   130
    John Johnson   133

=== Spot check: did Obama, Biden, Trump come back? ===
  Barack Obama         body_len=  83878
  Joe Biden            body_len=  96065
  Donald Trump         body_len=  76744
